In [2]:
from pathlib import Path
import json
import pandas as pd

BANDIT_RESULTS_DIR = Path("semgrep-results")

records = []

for issue_file in BANDIT_RESULTS_DIR.glob("*/*/semgrep_introduced_issues.json"):

    repo_dir = issue_file.parent.parent.name
    pr_dir = issue_file.parent.name

    owner_repo = repo_dir.split("__", 1)
    if len(owner_repo) == 2:
        owner, repo = owner_repo
    else:
        owner, repo = None, repo_dir

    pr_number = pr_dir.replace("pr_", "")

    with open(issue_file, "r", encoding="utf-8") as f:
        issues = json.load(f)

    records.append({
        "owner": owner,
        "repo": repo,
        "pr_number": pr_number,
        "issue_file": str(issue_file),
        "introduced_issue_count": len(issues),
        "issues": issues,
    })

pr_df = pd.DataFrame(records)
pr_df = pr_df.sort_values(["introduced_issue_count", "owner", "repo"], ascending=[False, True, True]).reset_index(drop=True)

pr_df.head()

,owner,repo,pr_number,issue_file,introduced_issue_count,issues
0,MontrealAI,AGI-Alpha-Agent-v0,1899,semgrep-results/MontrealAI__AGI-Alpha-Agent-v0...,23,[{'check_id': 'python.lang.security.audit.dyna...
1,wandb,weave,4549,semgrep-results/wandb__weave/pr_4549/semgrep_i...,22,[{'check_id': 'python.django.security.audit.cu...
2,wandb,weave,4607,semgrep-results/wandb__weave/pr_4607/semgrep_i...,20,[{'check_id': 'python.django.security.audit.cu...
3,Azure,azure-sdk-for-python,41852,semgrep-results/Azure__azure-sdk-for-python/pr...,17,[{'check_id': 'python.lang.security.use-defuse...
4,MontrealAI,AGI-Alpha-Agent-v0,2462,semgrep-results/MontrealAI__AGI-Alpha-Agent-v0...,15,[{'check_id': 'python.lang.security.insecure-h...


In [3]:
issue_rows = []

for _, row in pr_df.iterrows():

    for issue in row["issues"]:

        extra = issue.get("extra", {})
        metadata = extra.get("metadata", {})

        issue_rows.append({
            "owner": row["owner"],
            "repo": row["repo"],
            "pr_number": row["pr_number"],

            "rule_id": issue.get("check_id"),
            "message": extra.get("message"),

            "severity": extra.get("severity"),
            "confidence": metadata.get("confidence"),

            "cwe": metadata.get("cwe"),
            "owasp": metadata.get("owasp"),

            "file": issue.get("path"),
            "line": issue.get("start", {}).get("line"),
        })

semgrep_issue_df = pd.DataFrame(issue_rows)

semgrep_issue_df.head()

,owner,repo,pr_number,rule_id,message,severity,confidence,cwe,owasp,file,line
0,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/af_requests.py,87
1,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/demos/macro_sentinel/data_fee...,71
2,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,296
3,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,299
4,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.insecure-hash-algorithms....,Detected SHA1 hash algorithm which is consider...,WARNING,MEDIUM,[CWE-327: Use of a Broken or Risky Cryptograph...,"[A03:2017 - Sensitive Data Exposure, A02:2021 ...",alpha_factory_v1/demos/meta_agentic_agi/core/p...,77


In [6]:
semgrep_issue_df['file'].value_counts()

file
weave/trace_server/calls_query_builder/calls_query_builder.py    40
jac/jaclang/compiler/unitree.py                                  34
pygs/graphserver/graphdb.py                                      24
app.py                                                           24
btcrecover/btcrpass.py                                           14
                                                                 ..
templates/base.html                                               1
pygs/graphserver/ext/osm/osmdb.py                                 1
adconnection_gui.py                                               1
adconnection_app.py                                               1
app/blueprints/wizard/routes.py                                   1
Name: count, Length: 170, dtype: int64

In [9]:
test_mask = semgrep_issue_df["file"].str.contains("test", case=False, na=False)
semgrep_issue_df[test_mask][["file", "rule_id", "message"]].head(20)

,file,rule_id,message
209,test_runner/fixtures/neon_fixtures.py,python.lang.security.audit.subprocess-shell-tr...,Found 'subprocess' function 'run' with 'shell=...
210,test_runner/fixtures/neon_fixtures.py,python.lang.security.audit.subprocess-shell-tr...,Found 'subprocess' function 'run' with 'shell=...
211,test_runner/fixtures/neon_fixtures.py,python.lang.security.audit.subprocess-shell-tr...,Found 'subprocess' function 'run' with 'shell=...
212,test_runner/fixtures/neon_fixtures.py,python.lang.security.audit.subprocess-shell-tr...,Found 'subprocess' function 'run' with 'shell=...
213,test_runner/fixtures/neon_fixtures.py,python.lang.security.audit.subprocess-shell-tr...,Found 'subprocess' function 'run' with 'shell=...
361,libs/core/kiln_ai/adapters/eval/test_g_eval.py,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t..."
362,libs/core/kiln_ai/adapters/eval/test_g_eval.py,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t..."
363,libs/core/kiln_ai/adapters/eval/test_g_eval.py,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t..."
364,libs/core/kiln_ai/adapters/eval/test_g_eval.py,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t..."
365,libs/core/kiln_ai/adapters/eval/test_g_eval.py,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t..."


In [13]:
len(semgrep_issue_df[test_mask][["file", "rule_id", "message"]])

11

In [11]:
semgrep_issue_df_clean = semgrep_issue_df[
    ~semgrep_issue_df["file"].str.contains("test", case=False, na=False)
].copy()

semgrep_issue_df_clean

,owner,repo,pr_number,rule_id,message,severity,confidence,cwe,owasp,file,line
0,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/af_requests.py,87
1,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/demos/macro_sentinel/data_fee...,71
2,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,296
3,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,299
4,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.insecure-hash-algorithms....,Detected SHA1 hash algorithm which is consider...,WARNING,MEDIUM,[CWE-327: Use of a Broken or Risky Cryptograph...,"[A03:2017 - Sensitive Data Exposure, A02:2021 ...",alpha_factory_v1/demos/meta_agentic_agi/core/p...,77
...,...,...,...,...,...,...,...,...,...,...,...
566,ucbepic,docetl,379,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",docetl/operations/utils/api.py,1043
567,ucbepic,docetl,378,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",docetl/operations/utils/api.py,1043
568,vladfi1,slippi-ai,39,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t...",WARNING,LOW,[CWE-502: Deserialization of Untrusted Data],"[A08:2017 - Insecure Deserialization, A08:2021...",slippi_ai/rl/run_lib.py,568
569,winedarksea,AutoTS,259,python.lang.security.deserialization.pickle.av...,"Avoid using `pickle`, which is known to lead t...",WARNING,LOW,[CWE-502: Deserialization of Untrusted Data],"[A08:2017 - Insecure Deserialization, A08:2021...",autots/evaluator/auto_ts.py,2846


In [12]:
semgrep_issue_df_clean['confidence'].value_counts()

confidence
LOW       453
MEDIUM     82
HIGH       25
Name: count, dtype: int64

In [ ]:
semgrep_error_df = semgrep_issue_df_clean[
    semgrep_issue_df_clean["severity"] == "ERROR"
].copy()

print("Total ERROR issues:", len(semgrep_error_df))

error_prs = semgrep_error_df[
    ["owner", "repo", "pr_number"]
].drop_duplicates()

print("PRs with ERROR issues:", len(error_prs))

Total ERROR issues: 95
PRs with ERROR issues: 32


In [20]:
semgrep_error_df['cwe'].value_counts()

cwe
[CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')]    35
[CWE-611: Improper Restriction of XML External Entity Reference]                                        31
[CWE-601: URL Redirection to Untrusted Site ('Open Redirect')]                                          21
[CWE-295: Improper Certificate Validation]                                                               7
[CWE-300: Channel Accessible by Non-Endpoint]                                                            1
Name: count, dtype: int64